# Voice of the customer: what real Kuku reviews say

This notebook analyses a **real, public sample of Google Play reviews** for Kuku FM (audio) and Kuku TV (micro-drama). It is deliberately kept separate from the synthetic product analytics in `sql/` and `dashboard/` — review evidence is *observed public feedback*, not simulated metrics.

**Provenance.** Reviews were collected from Google Play's public reviews endpoint (India locale) with `notebooks/fetch_reviews.py`. User names were not stored — only an opaque id, star rating, text, date, app version and thumbs-up count. See `data/reviews_raw/collection_log.md` for the collection date, sample size and date range.

**Caveat stated up front.** Store reviews are *self-selected*: they skew to very satisfied or very dissatisfied users and are not a representative survey. Findings below are directional signals to investigate, not population estimates.

In [ ]:
import pandas as pd

labeled = pd.read_csv('../data/reviews_processed/kuku_reviews_labeled.csv')
themes  = pd.read_csv('../data/reviews_processed/theme_summary.csv')
print('Reviews:', len(labeled))
print('Date range:', labeled['review_date'].min(), 'to', labeled['review_date'].max())
print('By app:')
print(labeled.groupby('app')['rating'].agg(['count', 'mean']).round(2))

## Method

1. **Collection** — the most-recent public reviews per app, India store, English locale.
2. **Labelling** — each review is tagged with one or more *themes* using transparent keyword rules (`notebooks/classify_reviews.py`): payments & billing, pricing, content quality, playback/tech, ads, app stability, language, discovery/navigation, customer support, and generic praise. Rules are spot-checkable; there is no black-box model.
3. **Honesty** — reviews that match no theme (very short, or non-English) are kept as *uncategorised* and that share is reported, not hidden.

In [ ]:
# Rating distribution and the uncategorised share
dist = labeled.pivot_table(index='rating', columns='app', values='review_id',
                           aggfunc='count', fill_value=0)
print(dist)
uncat = (labeled['themes'].isna() | (labeled['themes'] == '')).mean() * 100
print(f'\nUncategorised (no theme matched): {uncat:.1f}%')

### Finding 1 — ratings are strongly polarised

The sample is bimodal: most reviews are 1-star or 5-star, with little in between. Kuku FM (audio) skews more negative (mean ~2.5) than Kuku TV (micro-drama, mean ~3.5). This is typical of self-selected store reviews and is why we read *themes*, not the average star rating.

![Rating distribution by app](../reports/figures/fig_rating_dist.png)

In [ ]:
# Theme volume and the average rating attached to each theme
themes.sort_values('mentions', ascending=False)

### Finding 2 — payments and autopay dominate the complaints

**Payments & billing is the single largest negative theme (~24% of all reviews, average rating ~1.2).** The recurring story is a ₹1 / low-cost trial that converts into an autopay charge users say they did not expect, plus difficulty cancelling and getting a refund. Pricing is a separate, also very low-rated theme. Content quality is mixed (average ~3.2) — it is discussed by both happy and unhappy users.

![Negative review themes](../reports/figures/fig_review_themes.png)

In [ ]:
# Evidence: among 1-star reviews, how many mention payments/billing?
one_star = labeled[labeled['rating'] == 1]
pay_1star = one_star['themes'].fillna('').str.contains('payments_billing').mean() * 100
print(f'1-star reviews mentioning payments/billing: {pay_1star:.1f}%')

# A few representative payment complaints (real, public; verbatim)
mask = labeled['themes'].fillna('').str.contains('payments_billing') & (labeled['rating'] == 1)
for t in labeled.loc[mask, 'text'].head(6):
    print('-', str(t)[:180])

## How this connects to the product funnel

The reviews corroborate the synthetic funnel analysis and point to the same place:

| Observed public feedback | Simulated analytical result | Product hypothesis | Validation needed |
|---|---|---|---|
| Payments/autopay is the #1 complaint (~24%, ~1.2 stars) | The paywall is the sharpest drop in the episode funnel; a simulated UPI-failure week dents conversion | A clearer trial-to-paid moment (consent, price, easy cancel) and payment-failure handling would lift both conversion *and* trust | Instrument the real paywall and payment funnel; confirm failure spikes against gateway logs |
| Pricing complaints cluster with payments | Later paywall (ep8 vs ep6) lifts subscribe +7pp in the simulated A/B | Reducing bill-shock may matter more than the headline price | A/B the trial-consent screen; measure refunds and churn, not just conversion |

**Next steps:** connect review themes to funnel steps with event-level tagging; track the payments theme over app versions to see whether fixes land; expand the sample across locales and languages.

*All star ratings and quotes above are real public reviews. All retention / funnel / revenue figures referenced are from a clearly labelled synthetic model and are not Kuku's real numbers.*